# CADF-Net v3: staged, resumable experiments

Use the **Python (histo)** kernel. The notebook is intentionally split into stages so you can work according to your available time.

Every training stage writes an atomic checkpoint at each epoch boundary and at least once per hour during long epochs. Re-running an interrupted stage cell resumes from its latest checkpoint. Feature/output extraction is also resumable.

Scientific guardrail: the fusion partition is used to fit feasibility/gating, validation is used for selection, and the held-out test cell should be executed only after Stages 1–5 are fixed. The 100-image smoke results are engineering checks, not research findings.

## 1. Locate Part 7 and import the implementation

In [1]:
from pathlib import Path
import json, sys, torch

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / 'experiment_parts' / 'part_07_cadf_net_v3',
]
PART_DIR = next(
    path.resolve() for path in candidates
    if (path / 'configuration' / 'cadfnet_v3.yaml').is_file()
)
sys.path.insert(0, str(PART_DIR / 'code'))

from cadfnet_v3.checkpoint import atomic_torch_save
from cadfnet_v3.config import load_config
from cadfnet_v3.pipeline import (
    export_evaluation, prepare_run, preflight_check,
    run_external_evaluation, run_trainable_ablation,
)
from cadfnet_v3.training import (
    cache_initial_features, collect_outputs, evaluate_ablation_variants,
    initialize_components, train_stage1_generalist,
    train_stage2_specialist, train_stage3_finetune,
    train_stage4_feasibility, train_stage5_gate, write_histories,
)
print('Part directory:', PART_DIR)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

c:\Users\Bilal Rehman\miniconda3\envs\histo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Part directory: C:\Users\Bilal Rehman\Desktop\Hard-Pair Research\experiment_parts\part_07_cadf_net_v3
PyTorch: 2.2.2+cu118 | CUDA: True


## 2. Choose full or smoke configuration

In [2]:
# Leave this False for the complete experiment. Set True only for a fresh engineering check.
SMOKE = False

config_path = PART_DIR / 'configuration' / 'cadfnet_v3.yaml'
override_path = PART_DIR / 'configuration' / 'smoke_overrides.yaml' if SMOKE else None
config = load_config(config_path, override_path)
frame, output_dir, device = prepare_run(config)

print('Mode:', 'SMOKE (100 images)' if SMOKE else 'FULL HAM10000')
print('Device:', device)
print('Output:', output_dir)
display(frame.groupby(['split', 'label']).size().unstack(fill_value=0))

Mode: FULL HAM10000
Device: cuda
Output: C:\Users\Bilal Rehman\Desktop\Hard-Pair Research\experiment_parts\part_07_cadf_net_v3\analysis_and_results\full


label,0,1,2,3,4,5,6
split,,,,,,,
fusion,38,66,101,10,111,651,15
test,35,55,93,4,125,675,23
train,212,340,792,96,759,4694,92
val,42,53,113,5,118,685,12


## 3. Initialize the model and run preflight checks

The shared encoder always receives five channels. The generalist view is normalized RGB plus two zero channels; the specialist view is lesion-crop RGB plus melanin and hemoglobin channels.

In [3]:
model, feasibility_predictor, gate, fusion = initialize_components(config)
preflight = preflight_check(model, frame, config, device)
print(json.dumps(preflight, indent=2))

{
  "generalist_view_shape": [
    2,
    5,
    192,
    192
  ],
  "specialist_view_shape": [
    2,
    5,
    192,
    192
  ],
  "color_stats_shape": [
    2,
    12
  ],
  "generalist_logits_shape": [
    2,
    7
  ],
  "specialist_logits_shape": [
    2,
    2
  ],
  "finite": true,
  "lesion_disjoint": true
}


## 4. Cache deterministic frozen-encoder features

This cache supports Stages 1–2. It checkpoints extraction progress hourly. Re-running this cell reuses a completed cache or resumes an interrupted one.

In [4]:
head_frame = frame[frame['split'].isin(['train', 'val'])].reset_index(drop=True)
feature_cache = cache_initial_features(
    model, head_frame, output_dir / 'initial_feature_cache.pt', config, device
)
print({key: tuple(value.shape) for key, value in feature_cache.items() if torch.is_tensor(value)})

{'generalist_features': (8013, 576), 'specialist_features': (8013, 576), 'color_stats': (8013, 12), 'labels': (8013,)}


## 5. Stage 1 — train the seven-class generalist head (15 full-run epochs)

In [5]:
histories = {}
histories['stage1'] = train_stage1_generalist(
    model, feature_cache, output_dir, config, device
)
histories['stage1'][-1]

{'epoch': 15.0,
 'loss': 0.32972681399917464,
 'macro_f1': 0.5138462944339114,
 'pair_conditional_correction_rate': 0.39361703395843506,
 'full_error_correction_potential': 0.43624159693717957}

## 6. Stage 2 — train the MEL/NV specialist head with error-focused loss (15 epochs)

In [6]:
histories['stage2'] = train_stage2_specialist(
    model, feature_cache, output_dir, config, device
)
histories['stage2'][-1]

{'epoch': 15.0,
 'loss': 0.035291127632369516,
 'macro_f1': 0.5287325202659154,
 'pair_conditional_correction_rate': 0.4193548262119293,
 'full_error_correction_potential': 0.5844155550003052}

## 7. Stage 3 — fine-tune the last three encoder blocks (10 epochs)

This is the most computationally expensive stage. Re-running after interruption resumes from the last hourly/epoch checkpoint.

In [7]:
train_frame = frame[frame['split'] == 'train'].reset_index(drop=True)
validation_frame = frame[frame['split'] == 'val'].reset_index(drop=True)
histories['stage3'] = train_stage3_finetune(
    model, train_frame, validation_frame, output_dir, config, device
)
histories['stage3'][-1]

{'epoch': 10.0,
 'loss': 0.40587789863121343,
 'generalist_loss': 0.3916635739231054,
 'specialist_loss': 0.04784629522258782,
 'macro_f1': 0.5415134425107194,
 'mel_sensitivity': 0.3474576271186441,
 'pair_conditional_correction_rate': 0.26966291666030884,
 'full_error_correction_potential': 0.49275362491607666}

## 8. Cache fusion-fitting and validation outputs

The held-out test split is deliberately not touched here.

In [8]:
fusion_frame = frame[frame['split'] == 'fusion'].reset_index(drop=True)
fusion_outputs = collect_outputs(
    model, fusion_frame, output_dir / 'fusion_outputs.pt', config, device
)
validation_outputs = collect_outputs(
    model, validation_frame, output_dir / 'validation_outputs.pt', config, device
)
print('Fusion images:', len(fusion_outputs['labels']), '| Validation images:', len(validation_outputs['labels']))

Fusion images: 992 | Validation images: 1028


## 9. Stage 4 — train the transfer-help feasibility predictor (50 epochs)

In [9]:
histories['stage4'] = train_stage4_feasibility(
    feasibility_predictor, fusion_outputs, validation_outputs,
    output_dir, config, device
)
histories['stage4'][-1]

{'epoch': 50.0,
 'loss': 0.44055841863155365,
 'validation_loss': 0.46704500913619995,
 'positive_mean': 0.3698556125164032,
 'negative_mean': 0.14039885997772217,
 'separation': 0.22945675253868103}

## 10. Stage 5 — train the differentiable alpha/beta gate (30 epochs)

In [10]:
histories['stage5'] = train_stage5_gate(
    feasibility_predictor, gate, fusion,
    fusion_outputs, validation_outputs, output_dir, config, device
)
write_histories(output_dir, histories)
histories['stage5'][-1]

{'epoch': 30.0,
 'loss': 0.6332063732608673,
 'validation_pair_balanced_accuracy': 0.6545032858848572,
 'mean_alpha': 0.25518637895584106,
 'mean_beta': 0.6479637622833252}

## 11. Freeze the selected model bundle

In [11]:
atomic_torch_save({
    'model_state': model.state_dict(),
    'feasibility_state': feasibility_predictor.state_dict(),
    'gate_state': gate.state_dict(),
    'config': config,
}, output_dir / 'cadfnet_v3_final.pt')
print('Saved:', output_dir / 'cadfnet_v3_final.pt')

Saved: C:\Users\Bilal Rehman\Desktop\Hard-Pair Research\experiment_parts\part_07_cadf_net_v3\analysis_and_results\full\cadfnet_v3_final.pt


## 12. Final held-out HAM10000 test evaluation — run once

Run this only after the architecture, losses, hyperparameters, and Stages 1–5 are fixed. It evaluates A0, A1, A3, and A4 without additional training.

In [12]:
test_frame = frame[frame['split'] == 'test'].reset_index(drop=True)
test_outputs = collect_outputs(
    model, test_frame, output_dir / 'test_outputs.pt', config, device
)
test_metrics, test_variants = evaluate_ablation_variants(
    test_outputs, feasibility_predictor, gate, fusion, config, device
)
export_evaluation(output_dir, test_outputs, test_metrics, test_variants, config)
display({name: {
    'pair_balanced_accuracy': values['pair_balanced_accuracy'],
    'mel_sensitivity': values['mel_sensitivity'],
    'nv_sensitivity': values['nv_sensitivity'],
    'macro_f1': values['macro_f1'],
} for name, values in test_metrics.items()})

{'A1_generalist_only': {'pair_balanced_accuracy': 0.6231111111111112,
  'mel_sensitivity': 0.384,
  'nv_sensitivity': 0.8622222222222222,
  'macro_f1': 0.5797740472129954},
 'A0_full': {'pair_balanced_accuracy': 0.6111111111111112,
  'mel_sensitivity': 0.36,
  'nv_sensitivity': 0.8622222222222222,
  'macro_f1': 0.5767617895355776},
 'A3_no_feasibility_transfer': {'pair_balanced_accuracy': 0.6111111111111112,
  'mel_sensitivity': 0.36,
  'nv_sensitivity': 0.8622222222222222,
  'macro_f1': 0.5781032270096936},
 'A4_fixed_gate': {'pair_balanced_accuracy': 0.6031111111111112,
  'mel_sensitivity': 0.344,
  'nv_sensitivity': 0.8622222222222222,
  'macro_f1': 0.5805321146336023}}

## 13. Optional retraining ablation A2 — standard specialist CE

This reuses the frozen Stage-1 generalist and writes to `ablations/A2`. It has independent hourly checkpoints.

In [13]:
RUN_A2 = False
if RUN_A2:
    a2_result = run_trainable_ablation(config, 'A2')
    display(a2_result['metrics'])
else:
    print('A2 skipped. Set RUN_A2=True when you have time.')

A2 skipped. Set RUN_A2=True when you have time.


## 14. Optional retraining ablation A5 — no dual view

This reuses the frozen Stage-1 generalist and writes to `ablations/A5`. It has independent hourly checkpoints.

In [ ]:
RUN_A5 = False
if RUN_A5:
    a5_result = run_trainable_ablation(config, 'A5')
    display(a5_result['metrics'])
else:
    print('A5 skipped. Set RUN_A5=True when you have time.')

## 15. Optional frozen BCN20000 evaluation

Run only after the final bundle is frozen. No BCN labels are used for fitting. SCC is excluded from the primary seven-class comparison and reported in the status file.

In [ ]:
RUN_BCN_EXTERNAL = True
if RUN_BCN_EXTERNAL:
    external_result = run_external_evaluation(config)
    display(external_result['metrics'])
else:
    print('BCN evaluation skipped. Set RUN_BCN_EXTERNAL=True after HAM decisions are fixed.')

## Checkpoint behavior

- `*_latest.pt`: resume state, updated hourly and at every epoch boundary.
- `*_best.pt`: best validation-selected state for that stage.
- `*.progress.pt`: temporary resumable extraction state.
- Completed cells are safe to rerun; completed artifacts are loaded instead of recomputed.

Do not delete `latest`, `best`, cache, or progress files if you want to resume.